# Qwen2.5-Math-1.5B Evaluation with MoTTT
This notebook evaluates the baseline `Qwen/Qwen2.5-Math-1.5B-Instruct` and compares it to `Qwen/Qwen2.5-Math-1.5B` enhanced with the MoTTT (Test-Time LoRA Scratchpad) method.

In [5]:
!nvidia-smi

In [7]:
!git clone https://github.com/Jerryliu3547/MoTTT.git /content/MoTTT
%cd /content/MoTTT
!pip install -q -e .

In [8]:
import os
import sys
from pathlib import Path

# Add src/ to path
SRC_DIR = Path().resolve().parent.parent / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from mottt.models.mottt_model import MoTTTModel


## 1. Evaluate Baseline (Qwen2.5-Math-1.5B-Instruct)

In [ ]:
baseline_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading {baseline_name}...")
baseline_tokenizer = AutoTokenizer.from_pretrained(baseline_name, trust_remote_code=True)
baseline_model = AutoModelForCausalLM.from_pretrained(
    baseline_name,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

baseline_pipe = pipeline(
    "text-generation",
    model=baseline_model,
    tokenizer=baseline_tokenizer,
    max_new_tokens=512,
    do_sample=False
)


In [ ]:
sample_context = "The treasure is hidden under the old oak tree in the center of the village. The village was founded in 1842."
sample_query = "If I walk 5 miles north from the village center, and then 3 miles east, how far am I from the treasure linearly?"

prompt = f"Background Context:\n{sample_context}\n\nQuestion:\n{sample_query}\n\nPlease solve the problem step by step and end your response with '#### [final numerical answer]'."

outputs = baseline_pipe(prompt)
print("Baseline Output:")
print("="*40)
print(outputs[0]["generated_text"][len(prompt):].strip())


## 2. Evaluate MoTTT Model (Qwen2.5-Math-1.5B + Query-Aware Router)

In [ ]:
# Note: In a real scenario, you'd load Qwen/Qwen2.5-Math-1.5B base model, 
# and load trained mottt_router.pt and reasoning_experts.pt from your checkpoint directory.

base_name = "Qwen/Qwen2.5-Math-1.5B"
print(f"Loading {base_name} for MoTTT...")
mottt_tokenizer = AutoTokenizer.from_pretrained(base_name, trust_remote_code=True)
hf_backbone = AutoModelForCausalLM.from_pretrained(
    base_name,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

mottt_model = MoTTTModel(
    hidden_dim=hf_backbone.config.hidden_size,
    num_reasoning_experts=4, # Assuming default config
    rank=16,
    alpha=16.0,
    base_backbone=hf_backbone,
    all_linear=True
).to(device)

# Example: Load your trained checkpoints here
# ckpt_dir = Path("../gsm8k/checkpoints")
# mottt_model.router.load_state_dict(torch.load(ckpt_dir / "mottt_router.pt"))
# Add loading logic for reasoning_experts.pt here


In [ ]:
# MoTTT Inner-Loop Adaptation (Dynamic Scratchpad)
mottt_model.reset_scratchpad()
mottt_model.set_routing_gates(None)

inner_opt = torch.optim.SGD(mottt_model.get_scratchpad_parameters(), lr=1e-3)
inner_steps = 1

ctx_enc = mottt_tokenizer(sample_context, truncation=True, max_length=256, return_tensors="pt").to(device)

print("Running test-time inner loop adaptation on distractor context...")
for _ in range(inner_steps):
    ctx_out = hf_backbone(input_ids=ctx_enc.input_ids, attention_mask=ctx_enc.attention_mask)
    shift_logits = ctx_out.logits[..., :-1, :].contiguous()
    shift_labels = ctx_enc.input_ids[..., 1:].contiguous()
    inner_loss = F.cross_entropy(shift_logits.view(-1, hf_backbone.config.vocab_size), shift_labels.view(-1))
    
    inner_opt.zero_grad()
    inner_loss.backward()
    inner_opt.step()
    
print("Inner loop complete. Scratchpad adapted.")


In [ ]:
# Query-Aware Routing & Generation
q_enc = mottt_tokenizer(sample_query, return_tensors="pt").to(device)

with torch.no_grad():
    q_emb = hf_backbone.model.embed_tokens(q_enc.input_ids).mean(dim=1)
    # The router expects shape: (batch_size, seq_len, hidden_dim) and (batch_size, hidden_dim)
    gates, _ = mottt_model.router(q_emb.unsqueeze(0).unsqueeze(0), q_emb.unsqueeze(0))

mottt_model.set_routing_gates(gates)
print(f"Router Gates: {gates.mean(dim=0).mean(dim=0).tolist()}")

# Create pipeline with adapted backbone
mottt_pipe = pipeline(
    "text-generation",
    model=hf_backbone,
    tokenizer=mottt_tokenizer,
    max_new_tokens=512,
    do_sample=False
)

outputs = mottt_pipe(prompt)
print("\nMoTTT Output:")
print("="*40)
print(outputs[0]["generated_text"][len(prompt):].strip())

# Clean up
mottt_model.reset_scratchpad()
